**Problem Statement:**

You are given a dataset that contains user activity logs, where each row represents a date a user logged into an app.

Write a PySpark program to identify gaps in login activity per user — i.e., the date ranges when the user was inactive.

✅ Input Columns:

user_id: String (e.g., "U001")

login_date: Date (e.g., "2024-01-01")

Assume:
Dates are not continuous.
Each user is expected to be active daily.

✅ Expected Output Columns:
user_id

inactive_from_date

inactive_to_date
Only output gaps of more than 1 day.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("MissingDataRanges").getOrCreate()

In [0]:
#Sample Indian Login Data
data=[("U001","2024-01-01"),
("U001","2024-01-02"),
("U001","2024-01-05"),
("U001","2024-01-06"),
("U002","2024-01-03"),
("U002","2024-01-07")
]

df = spark.createDataFrame(data,["user_id","login_date"])
display(df)

In [0]:
df = df.withColumn("login_date",F.col("login_date").cast("date"))

In [0]:
windowSpec = Window.partitionBy("user_id").orderBy("login_date")
df = df.withColumn("prev",F.lag("login_date").over(windowSpec))
display(df)

In [0]:
gap_df = df.withColumn("gap_days",F.datediff(F.col("login_date"),F.col("prev")))
gap_df = gap_df.filter(F.col("gap_days")>1)
display(gap_df)

In [0]:
final_result =  gap_df.select("user_id",F.col("prev").alias("inactive_from_date"),F.col("login_date").alias("inactive_to_date"))

In [0]:
display(final_result)